### Library and function imports

In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Data processing

In [78]:
df = pd.read_csv("../dataset.csv")

# drop the empty column
df = df.iloc[:, [x for x in range(df.shape[1]) if x != 26]]

# copy responses from the non-vinunian branch to columns of the vinunian branch
df.iloc[df.iloc[:, 3] == "Non-VinUnian", 4:25] = df.iloc[
    df.iloc[:, 3] == "Non-VinUnian", 26:
].values
# df.iloc[df.iloc[:, 3] == "Non-VinUnian", 3] = df.iloc[
#     df.iloc[:, 3] == "Non-VinUnian", 25
# ]
df = df.iloc[:, :25]

# rename the columns
df.columns = [
    "timestamp",
    "age",
    "gender",
    "college",
    "year_of_study",
    "gpa_range",
    "currently_employed",
    "leisure_hours",
    "self_study_hours",
    "study_environment",
    "sleep_hours",
    "impact_on_learning_progress",
    "far_away_deadlines",
    "study_plan",
    "start_early_when_necessary",
    "difficult_to_maintain",
    "data_quality_check",
    "comfort_or_convenience",
    "peers_consider_normal",
    "control_over_habit",
    "sooner_next_month",
    "delay_until_deadline",
    "focus_frequency",
    "delay_unenjoyable_assignments",
    "referrer",
]
# filter out wrong answers to the data quality question
df = df[df["data_quality_check"] == "Somewhat Disagree"]

# filter out unrealistic numerical answers
df = df[
    (df["leisure_hours"] <= 12)
    & (df["self_study_hours"] <= 12)
    & (df["sleep_hours"] <= 12)
]

# merge options in study_environment feature
study_environment_map = {
    "Private space (i.e. bedroom, home office)": "Private Space",
    "Open learning space (i.e. university library)": "Open Space",
    "Collaborative space (i.e. group study room)": "Collab Space",
    "Cafes": "Flexible/Multiple",
    "Dormitory": "Private Space",
    "Any quiet space": "Private Space",
    "Flexible": "Flexible/Multiple",
    "Flexible, could be private place or open learning space": "Flexible/Multiple",
    "All above": "Flexible/Multiple",
    "all places above, and all places possible like pantry rooms, JA102, etc": "Flexible/Multiple",
}
df["study_environment"] = df["study_environment"].map(study_environment_map)

# convert timestamp to datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], dayfirst=True)

# code to publish a cleaned version of dataset
df.to_csv("../cleaned_dataset.csv", index=False)

df.head()

,timestamp,age,gender,college,year_of_study,gpa_range,currently_employed,leisure_hours,self_study_hours,study_environment,...,difficult_to_maintain,data_quality_check,comfort_or_convenience,peers_consider_normal,control_over_habit,sooner_next_month,delay_until_deadline,focus_frequency,delay_unenjoyable_assignments,referrer
0,2026-05-14 00:28:00,19,Male,CECS (Engineering & Computer Science),1st Year,3.5 – 4.0,No,2.0,4.0,Private Space,...,Somewhat Agree,Somewhat Disagree,Agree,Somewhat Agree,Somewhat Agree,Neutral,Somewhat Agree,5.0,4.0,Nguyễn Cảnh Kỳ
1,2026-05-14 00:30:30,19,Female,CBM (Business and Management),1st Year,3.5 – 4.0,Yes,2.0,8.0,Private Space,...,Strongly Disagree,Somewhat Disagree,Strongly Disagree,Somewhat Disagree,Neutral,Strongly Agree,Disagree,4.0,5.0,Đặng Tuấn Kiệt
2,2026-05-14 00:30:32,19,Male,CECS (Engineering & Computer Science),1st Year,3.0 – 3.49,No,4.0,3.0,Open Space,...,Disagree,Somewhat Disagree,Somewhat Disagree,Somewhat Disagree,Neutral,Neutral,Disagree,3.0,4.0,Đặng Tuấn Kiệt
4,2026-05-14 00:32:25,19,Male,CECS (Engineering & Computer Science),1st Year,3.0 – 3.49,No,5.0,4.0,Private Space,...,Neutral,Somewhat Disagree,Strongly Agree,Somewhat Disagree,Neutral,Strongly Disagree,Neutral,5.0,6.0,Nguyễn Trọng Nhân
5,2026-05-14 00:33:32,23,Male,CHS (Health Science),1st Year,3.5 – 4.0,No,4.0,1.0,Private Space,...,Agree,Somewhat Disagree,Agree,Agree,Somewhat Agree,Neutral,Agree,5.0,6.0,Bill


In [79]:
df["study_environment"].unique()

<StringArray>
['Private Space', 'Open Space', 'Flexible/Multiple', 'Collab Space', nan]
Length: 5, dtype: str

In [80]:
df["college"].unique()

<StringArray>
['CECS (Engineering & Computer Science)',
         'CBM (Business and Management)',
                  'CHS (Health Science)',
               'CAS (Arts and Sciences)',
                          'Non-VinUnian']
Length: 5, dtype: str

## **Encoding**

In [81]:
# =====================================================
# ENCODING: ORDINAL vs NOMINAL FEATURES
# =====================================================

# ORDINAL ENCODING (features with natural ordering)
likert_map = {
    "Strongly Disagree": 1,
    "Disagree": 2,
    "Somewhat Disagree": 3,
    "Neutral": 4,
    "Somewhat Agree": 5,
    "Agree": 6,
    "Strongly Agree": 7,
}

year_of_study_map = {
    "1st Year": 1,
    "2nd Year": 2,
    "3rd Year": 3,
    "4th Year": 4,
    "5th Year": 5,
    "6th Year": 6,
}

gpa_map = {
    "< 2.0": 1,
    "2.0 – 2.49": 2,
    "2.5 – 2.99": 3,
    "3.0 – 3.49": 4,
    "3.5 – 4.0": 5,
}

employ_map = {"Yes": 1, "No": 0}

# Apply ordinal encoding to Likert-scale features
likert_columns = [
    "impact_on_learning_progress",
    "far_away_deadlines",
    "study_plan",
    "start_early_when_necessary",
    "difficult_to_maintain",
    "comfort_or_convenience",
    "peers_consider_normal",
    "control_over_habit",
    "sooner_next_month",
    "delay_until_deadline",
    "focus_frequency",
    "delay_unenjoyable_assignments",
    "data_quality_check",
]

for col in likert_columns:
    df[col] = df[col].map(likert_map)

# Apply ordinal encoding to ordinal features
df["year_of_study"] = df["year_of_study"].map(year_of_study_map)
df["gpa_range"] = df["gpa_range"].map(gpa_map)
df["currently_employed"] = df["currently_employed"].map(employ_map)

print("✓ Ordinal encoding applied to ordinal features")

# =====================================================
# ONE-HOT ENCODING (nominal categorical features)
# =====================================================

# One-hot encode nominal categorical features
nominal_features = ["gender", "college", "study_environment"]

df = pd.get_dummies(df, columns=nominal_features, drop_first=False)
df = df.replace({False : 0, True : 1})
print("✓ One-hot encoding applied to nominal features")
print(f"Dataset shape after encoding: {df.shape}")
print(f"\nNew columns created:")
print([col for col in df.columns if any(feat in col for feat in nominal_features)])

# Save encoded dataset
df.to_csv("../encoded_dataset.csv", index=False)
print("\n✓ Encoded dataset saved to encoded_dataset.csv")

df.head()


✓ Ordinal encoding applied to ordinal features
✓ One-hot encoding applied to nominal features
Dataset shape after encoding: (193, 35)

New columns created:
['gender_Female', 'gender_Male', 'gender_Other', 'gender_Prefer not to say', 'college_CAS (Arts and Sciences)', 'college_CBM (Business and Management)', 'college_CECS (Engineering & Computer Science)', 'college_CHS (Health Science)', 'college_Non-VinUnian', 'study_environment_Collab Space', 'study_environment_Flexible/Multiple', 'study_environment_Open Space', 'study_environment_Private Space']

✓ Encoded dataset saved to encoded_dataset.csv


,timestamp,age,year_of_study,gpa_range,currently_employed,leisure_hours,self_study_hours,sleep_hours,impact_on_learning_progress,far_away_deadlines,...,gender_Prefer not to say,college_CAS (Arts and Sciences),college_CBM (Business and Management),college_CECS (Engineering & Computer Science),college_CHS (Health Science),college_Non-VinUnian,study_environment_Collab Space,study_environment_Flexible/Multiple,study_environment_Open Space,study_environment_Private Space
0,2026-05-14 00:28:00,19,1,5,0,2.0,4.0,8.0,5,5,...,0,0,0,1,0,0,0,0,0,1
1,2026-05-14 00:30:30,19,1,5,1,2.0,8.0,7.0,7,6,...,0,0,1,0,0,0,0,0,0,1
2,2026-05-14 00:30:32,19,1,4,0,4.0,3.0,6.0,5,6,...,0,0,0,1,0,0,0,0,1,0
4,2026-05-14 00:32:25,19,1,4,0,5.0,4.0,7.0,7,6,...,0,0,0,1,0,0,0,0,0,1
5,2026-05-14 00:33:32,23,1,5,0,4.0,1.0,5.0,6,7,...,0,0,0,0,1,0,0,0,0,1


In [82]:
# linkert_map = {
#     "Strongly Disagree": 1,
#     "Disagree": 2,
#     "Somewhat Disagree": 3,
#     "Neutral": 4,
#     "Somewhat Agree": 5,
#     "Agree": 6,
#     "Strongly Agree": 7,
# }

# df = df.replace(linkert_map)

In [83]:
# gender_map = {"Male": 1, "Female": 2, "Other": 3, "Prefer not to say": 4}

# college_map = {
#     "CECS (Engineering & Computer Science)": 1,
#     "CBM (Business and Management)": 2,
#     "CHS (Health Science)": 3,
#     "CAS (Arts and Sciences)": 4,
#     "Non-VinUnian": 5,
# }

# year_of_study_map = {
#     "1st Year": 1,
#     "2nd Year": 2,
#     "3rd Year": 3,
#     "4th Year": 4,
#     "5th Year": 5,
#     "6th Year": 6,
# }

# gpa_map = {
#     "< 2.0": 1,
#     "2.0 – 2.49": 2,
#     "2.5 – 2.99": 3,
#     "3.0 – 3.49": 4,
#     "3.5 – 4.0": 5,
# }

# employ_map = {"Yes": 1, "No": 0}

# study_env_map = {
#     "Private Space": 1,
#     "Open Space": 2,
#     "Flexible/Multiple": 3,
#     "Collab Space": 4,
# }

# df["gender"] = df["gender"].map(gender_map)
# df["college"] = df["college"].map(college_map)
# df["year_of_study"] = df["year_of_study"].map(year_of_study_map)
# df["gpa_range"] = df["gpa_range"].map(gpa_map)
# df["currently_employed"] = df["currently_employed"].map(employ_map)
# df["study_environment"] = df["study_environment"].map(study_env_map)

In [84]:
# df.head()

In [85]:
# df.to_csv("../encoded_dataset.csv", index=False)